In [8]:
# 1. Imports and global configuration

import os
import math
import time
import json
import random
from dataclasses import dataclass
from typing import Dict

import numpy as np
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


In [9]:
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, random_split

@dataclass
class DataConfig:
    dataset_name: str = "KMNIST"
    data_dir: str = "." # Changed from "./data" to "." for colab default behavior
    batch_size: int = 128
    val_fraction: float = 0.15
    num_workers: int = 0


def load_dataset(cfg: DataConfig):

    dataset_name = ""

    try:

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

        train_ds = datasets.KMNIST(
            root=cfg.data_dir,
            train=True,
            download=True,
            transform=transform
        )

        test_ds = datasets.KMNIST(
            root=cfg.data_dir,
            train=False,
            download=True,
            transform=transform
        )

        input_dim = 28 * 28
        num_classes = 10

        print("Dataset: KMNIST")
        dataset_name = "KMNIST"

    except Exception as e:

        print("KMNIST download failed. Using sklearn digits fallback.")
        print(e)

        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split

        digits = load_digits()

        X = digits.data.astype(np.float32)
        y = digits.target.astype(np.int64)

        X = X / (X.max() + 1e-8)

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=SEED,
            stratify=y
        )

        train_ds = TensorDataset(
            torch.from_numpy(X_train),
            torch.from_numpy(y_train)
        )

        test_ds = TensorDataset(
            torch.from_numpy(X_test),
            torch.from_numpy(y_test)
        )

        input_dim = X.shape[1]
        num_classes = 10
        dataset_name = "Sklearn Digits"

    val_size = int(len(train_ds) * cfg.val_fraction)
    train_size = len(train_ds) - val_size

    generator = torch.Generator().manual_seed(SEED)

    train_split, val_split = random_split(
        train_ds,
        [train_size, val_size],
        generator=generator
    )

    train_loader = DataLoader(train_split, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_split, batch_size=cfg.batch_size)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size)

    return train_loader, val_loader, test_loader, input_dim, num_classes, dataset_name


cfg = DataConfig()
train_loader, val_loader, test_loader, INPUT_DIM, NUM_CLASSES, DATASET_NAME = load_dataset(cfg)

print("INPUT_DIM:", INPUT_DIM)
print("NUM_CLASSES:", NUM_CLASSES)
print("DATASET_NAME:", DATASET_NAME)


KMNIST download failed. Using sklearn digits fallback.
Error downloading train-images-idx3-ubyte.gz:
Tried http://codh.rois.ac.jp/kmnist/dataset/kmnist/, got:
<urlopen error [Errno 110] Connection timed out>

INPUT_DIM: 64
NUM_CLASSES: 10
DATASET_NAME: Sklearn Digits


In [10]:
# 3. MLP model with Dropout and BatchNorm

class MLP(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes,
        hidden1=256,
        hidden2=128,
        dropout=0.0,
        batchnorm=False
    ):
        super().__init__()

        layers = []

        layers.append(nn.Linear(input_dim, hidden1))

        if batchnorm:
            layers.append(nn.BatchNorm1d(hidden1))

        layers.append(nn.ReLU())

        if dropout > 0:
            layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(hidden1, hidden2))

        if batchnorm:
            layers.append(nn.BatchNorm1d(hidden2))

        layers.append(nn.ReLU())

        if dropout > 0:
            layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(hidden2, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):

        if x.dim() > 2:
            x = x.view(x.size(0), -1)

        return self.net(x)


In [11]:
# 4. Training utilities

criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer):

    model.train()

    total_loss = 0
    total_correct = 0
    total_seen = 0

    for x, y in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        bs = y.size(0)

        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total_seen += bs

    return total_loss / total_seen, total_correct / total_seen


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_seen = 0

    for x, y in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)

        loss = criterion(logits, y)

        bs = y.size(0)

        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total_seen += bs

    return total_loss / total_seen, total_correct / total_seen


In [12]:
# 5. EarlyStopping

class EarlyStopping:

    def __init__(self, patience=3):

        self.patience = patience
        self.best = None
        self.counter = 0

    def step(self, val_loss):

        if self.best is None or val_loss < self.best:

            self.best = val_loss
            self.counter = 0
            return False

        self.counter += 1

        return self.counter >= self.patience

In [13]:
# 6. Optimizer factory

def make_optimizer(model, kind, lr, momentum=0.9, weight_decay=0.0):

    kind = kind.lower()

    if kind == "adam":

        return torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

    if kind == "sgd":

        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )


In [14]:
# 7. Training loop

def fit(model, optimizer, epochs=6, early_stop=False):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    stopper = EarlyStopping()

    for epoch in range(epochs):

        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer)
        va_loss, va_acc = evaluate(model, val_loader)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        print(
            f"Epoch {epoch+1} | train {tr_loss:.3f} | val {va_loss:.3f} | acc {va_acc:.3f}"
        )

        if early_stop:

            if stopper.step(va_loss):

                print("Early stopping triggered")
                break

    return history

In [15]:
# 8. Experiment runner

def run_experiment(
    exp_id,
    optimizer_kind="adam",
    lr=1e-3,
    dropout=0.0,
    batchnorm=False,
    weight_decay=0.0,
    momentum=0.9,
    epochs=6,
    early_stop=False
):

    model = MLP(
        INPUT_DIM,
        NUM_CLASSES,
        dropout=dropout,
        batchnorm=batchnorm
    ).to(DEVICE)

    # Generate a simple model summary. For more detailed summary, torchsummary could be used.
    model_summary = str(model)

    optimizer = make_optimizer(
        model,
        optimizer_kind,
        lr,
        momentum,
        weight_decay
    )

    history = fit(model, optimizer, epochs, early_stop)

    best_val_acc = float(np.max(history["val_acc"]))
    best_val_loss = float(np.min(history["val_loss"]))

    return {
        "exp_id": exp_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "optimizer": optimizer_kind,
        "lr": lr,
        "dropout": dropout,
        "batchnorm": batchnorm,
        "weight_decay": weight_decay,
        "momentum": momentum,
        "epochs_ran": len(history["val_acc"]),
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "model_summary": model_summary,
        "history": history,
        "model": model
    }


In [16]:
# 9. Run experiments

results = []

results.append(run_experiment("E1", "adam", 1e-3))

results.append(run_experiment("E2", "adam", 1e-3, dropout=0.3))

results.append(run_experiment("E3", "adam", 1e-3, batchnorm=True))

results.append(run_experiment("E4", "adam", 1e-3, dropout=0.3, early_stop=True, epochs=20))

results.append(run_experiment("O1", "adam", 1e-1))

results.append(run_experiment("O2", "adam", 1e-5))

results.append(run_experiment("O3", "sgd", 0.1, momentum=0.9, weight_decay=1e-4))



Epoch 1 | train 2.227 | val 2.111 | acc 0.721
Epoch 2 | train 1.979 | val 1.763 | acc 0.819
Epoch 3 | train 1.567 | val 1.261 | acc 0.874
Epoch 4 | train 1.079 | val 0.788 | acc 0.893
Epoch 5 | train 0.705 | val 0.486 | acc 0.930
Epoch 6 | train 0.492 | val 0.340 | acc 0.944
Epoch 1 | train 2.246 | val 2.131 | acc 0.730
Epoch 2 | train 2.024 | val 1.812 | acc 0.800
Epoch 3 | train 1.667 | val 1.350 | acc 0.819
Epoch 4 | train 1.232 | val 0.889 | acc 0.902
Epoch 5 | train 0.894 | val 0.581 | acc 0.902
Epoch 6 | train 0.666 | val 0.417 | acc 0.930
Epoch 1 | train 1.510 | val 1.949 | acc 0.847
Epoch 2 | train 0.686 | val 1.278 | acc 0.949
Epoch 3 | train 0.432 | val 0.656 | acc 0.981
Epoch 4 | train 0.293 | val 0.333 | acc 0.986
Epoch 5 | train 0.210 | val 0.207 | acc 0.986
Epoch 6 | train 0.158 | val 0.160 | acc 0.986
Epoch 1 | train 2.245 | val 2.136 | acc 0.623
Epoch 2 | train 2.020 | val 1.828 | acc 0.712
Epoch 3 | train 1.649 | val 1.346 | acc 0.805
Epoch 4 | train 1.206 | val 0.885 

In [17]:
# 10. Results table

rows = []

for r in results:

    rows.append({
        "exp_id": r["exp_id"],
        "dataset": r["dataset"],
        "seed": r["seed"],
        "optimizer": r["optimizer"],
        "lr": r["lr"],
        "dropout": r["dropout"],
        "batchnorm": r["batchnorm"],
        "weight_decay": r["weight_decay"],
        "momentum": r["momentum"],
        "epochs_ran": r["epochs_ran"],
        "best_val_acc": r["best_val_acc"],
        "best_val_loss": r["best_val_loss"],
        "model_summary": r["model_summary"]
    })


df = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False)

print(df)


  exp_id         dataset  seed optimizer       lr  dropout  batchnorm  \
2     E3  Sklearn Digits    42      adam  0.00100      0.0       True   
3     E4  Sklearn Digits    42      adam  0.00100      0.3      False   
6     O3  Sklearn Digits    42       sgd  0.10000      0.0      False   
0     E1  Sklearn Digits    42      adam  0.00100      0.0      False   
1     E2  Sklearn Digits    42      adam  0.00100      0.3      False   
4     O1  Sklearn Digits    42      adam  0.10000      0.0      False   
5     O2  Sklearn Digits    42      adam  0.00001      0.0      False   

   weight_decay  momentum  epochs_ran  best_val_acc  best_val_loss  \
2        0.0000       0.9           6      0.986047       0.159643   
3        0.0000       0.9          20      0.986047       0.093851   
6        0.0001       0.9           6      0.962791       0.147888   
0        0.0000       0.9           6      0.944186       0.340033   
1        0.0000       0.9           6      0.930233       0.41724

In [18]:
# 11. Save artifacts

os.makedirs("artifacts", exist_ok=True)


df.to_csv("artifacts/runs.csv", index=False)

best_exp = max(results, key=lambda x: x["best_val_acc"])


torch.save(best_exp["model"].state_dict(), "artifacts/best_model.pt")

best_config = {
    "optimizer": best_exp["optimizer"],
    "lr": best_exp["lr"],
    "dropout": best_exp["dropout"],
    "batchnorm": best_exp["batchnorm"],
    "weight_decay": best_exp["weight_decay"],
    "momentum": best_exp["momentum"],
    "dataset": best_exp["dataset"],
    "seed": best_exp["seed"]
}

with open("artifacts/best_config.json", "w") as f:

    json.dump(best_config, f, indent=4)


print("Saved artifacts to ./artifacts")


Saved artifacts to ./artifacts


In [19]:
import matplotlib.pyplot as plt
import os

os.makedirs("./artifacts/figures", exist_ok=True)


def plot_curves(history, title, save_path):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.plot(epochs, history["train_loss"], label="train")
    plt.plot(epochs, history["val_loss"], label="val")
    plt.title("Loss")
    plt.xlabel("epoch")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, history["train_acc"], label="train")
    plt.plot(epochs, history["val_acc"], label="val")
    plt.title("Accuracy")
    plt.xlabel("epoch")
    plt.legend()

    plt.suptitle(title)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


# лучший прогон
best_exp = max(results, key=lambda x: x["best_val_acc"])

plot_curves(
    best_exp["history"],
    f"Best run: {best_exp['exp_id']}",
    "./artifacts/figures/curves_best.png"
)


# плохие learning rates
lr_big = next(x for x in results if x["exp_id"] == "O1")
lr_small = next(x for x in results if x["exp_id"] == "O2")

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(lr_big["history"]["val_loss"], label="lr=0.1")
plt.plot(lr_small["history"]["val_loss"], label="lr=1e-5")
plt.title("Validation Loss (LR extremes)")
plt.legend()

plt.subplot(1,2,2)
plt.plot(lr_big["history"]["val_acc"], label="lr=0.1")
plt.plot(lr_small["history"]["val_acc"], label="lr=1e-5")
plt.title("Validation Accuracy (LR extremes)")
plt.legend()

plt.tight_layout()
plt.savefig("./artifacts/figures/curves_lr_extremes.png")
plt.close()

print("Figures saved to ./artifacts/figures/")

Figures saved to ./artifacts/figures/
